In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import joblib
import matplotlib.pyplot as plt

In [2]:
RAW_PATH = '../dataset/IMS_Data_200725-200726.csv'

PROCESSED_DIR = Path('../dataset/DeepAR processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_csv(RAW_PATH)

df = df[df['QualityStatus'] == 'Good'].copy()

df = df[['DateTime', 'Value']]
df.columns = ['time', 'pressure']

df['time'] = pd.to_datetime(df['time'])
df['pressure'] = pd.to_numeric(df['pressure'], errors='coerce')

df = df.dropna().sort_values('time')

print(df.shape)

(1177, 2)


C:\Users\Zepunnn\AppData\Local\Temp\ipykernel_30304\1064938457.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time'] = pd.to_datetime(df['time'])


In [4]:
INVALID_START = pd.Timestamp('2025-08-01')
INVALID_END   = pd.Timestamp('2026-04-30 23:59:59')

df = df[
    ~((df['time'] >= INVALID_START) &
      (df['time'] <= INVALID_END))
].copy()

print(df.shape)

(308, 2)


In [5]:
df = (
    df.set_index('time')
      .resample('1min')
      .mean()
)

# interpolasi gap kecil
df['pressure'] = df['pressure'].interpolate(
    method='time',
    limit=30
)

df = df.dropna().reset_index()

print(df.shape)

(6128, 2)


In [6]:
TRAIN_END = pd.Timestamp('2026-06-30 23:59:59')
EVAL_START = pd.Timestamp('2026-07-01')
EVAL_END   = pd.Timestamp('2026-07-31 23:59:59')

train_df = df[df['time'] <= TRAIN_END].copy()

eval_df = df[
    (df['time'] >= EVAL_START) &
    (df['time'] <= EVAL_END)
].copy()

print('Train:', train_df.shape)
print('Eval :', eval_df.shape)

Train: (4805, 2)
Eval : (1323, 2)


In [7]:
scaler = StandardScaler()

train_df['pressure'] = scaler.fit_transform(
    train_df[['pressure']]
)

eval_df['pressure'] = scaler.transform(
    eval_df[['pressure']]
)

joblib.dump(scaler, PROCESSED_DIR / 'scaler.pkl')

['..\\dataset\\DeepAR processed\\scaler.pkl']

In [8]:
train_df['series_id'] = 'pressure_sensor'
eval_df['series_id'] = 'pressure_sensor'

# time index kontinu
full_df = pd.concat([train_df, eval_df]).sort_values('time')

full_df['time_idx'] = np.arange(len(full_df))

# split kembali
train_df = full_df[full_df['time'] <= TRAIN_END].copy()
eval_df = full_df[
    (full_df['time'] >= EVAL_START) &
    (full_df['time'] <= EVAL_END)
].copy()

train_df.head()

,time,pressure,series_id,time_idx
0,2025-07-20 00:00:00,-0.476599,pressure_sensor,0
1,2025-07-20 00:01:00,-0.478171,pressure_sensor,1
2,2025-07-20 00:02:00,-0.479743,pressure_sensor,2
3,2025-07-20 00:03:00,-0.481315,pressure_sensor,3
4,2025-07-20 00:04:00,-0.482887,pressure_sensor,4


In [9]:
train_df.to_csv(PROCESSED_DIR / 'train.csv', index=False)
eval_df.to_csv(PROCESSED_DIR / 'eval.csv', index=False)

eval_df[['time']].to_csv(
    PROCESSED_DIR / 'eval_timestamps.csv',
    index=False
)

print('Saved:')
print('- train.csv')
print('- eval.csv')
print('- scaler.pkl')
print('- eval_timestamps.csv')

Saved:
- train.csv
- eval.csv
- scaler.pkl
- eval_timestamps.csv
